# SDF Objects in the Standard Viewer

**Part I · Visualization** — Tutorial 05

The standard mesh viewer can also render smooth, **ray-marched**
signed-distance-field (SDF) solids — mixed with normal meshes in the same
scene. You will learn to:

- Opt a single entity into SDF rendering with the `SdfStyle` marker.
- Tune per-entity SDF styles (`SdfSphereStyle`, `SdfLineStyle`, …).
- Compose solids with `SdfObject` + the Python CSG operators (`+`/`-`/`&`/`^`).
- Bundle constituents with `Composed`, and group members into one solid with
  `SdfGroup`.

> **Note:** SDF objects require **WebGL2**. On WebGL1 they are skipped and a
> single warning banner is shown. SDF objects self-shadow within an
> object/group but do **not** cast/receive shadows onto *other* scene objects.


## Setup


In [1]:
from pytanga.geometry import (
    Box, Cylinder, Direction, Disk, Ellipse, Ellipsoid, Line, PartialDisk, Point, Sphere,
    regular_polygon,
)
from pytanga.viz import (
    SdfBoxStyle, SdfCircleStyle, SdfCylinderStyle, SdfDiskStyle, SdfEllipseStyle,
    SdfEllipsoidStyle, SdfLineStyle, SdfPartialDiskStyle, SdfPointStyle,
    SdfRegularPolygonStyle, SdfSphereStyle, SdfStyle, Visualizer,
)
from pytanga.viz.sdf import Composed, SdfGroup, SdfObject, capped_cylinder, sphere


## 1. The `SdfStyle` marker

Pass `style=SdfStyle(...)` to `add()` to opt **that one entity** into
ray-marched SDF rendering (`kind:"sdf"` on the wire) instead of the normal mesh
renderer. `color`/`opacity` still resolve through the normal priority chain;
the remaining fields are SDF-specific knobs (`soft_shadows`, `max_steps`,
`bound_padding`, `antialias`).


In [2]:
viz = Visualizer(title="SDF — mesh vs ray-marched", add_default_axes=False, add_default_grid=False)
viz.add(Sphere(Point(-2.5, 0, 0), 1.0), color="#4477cc", label="mesh")              # normal mesh
viz.add(Sphere(Point(0, 0, 0), 1.1), style=SdfStyle(color="#ffaa00"), label="SDF")   # ray-marched
viz.display_snapshot()


## 2. Per-entity SDF styles

Each entity kind has a dedicated SDF style class that inherits the `SdfStyle`
knobs and adds a per-entity parameter (`thickness`, `tube_radius`, `size`, …).


In [3]:
viz = Visualizer(title="SDF — per-entity styles", add_default_axes=False, add_default_grid=False)

viz.add(Sphere(Point(-3, 0, 0), 1.0), style=SdfSphereStyle(color="#ffaa00"))
viz.add(
    Line.from_points(Point(-1.5, -1, 0), Point(-1.5, 1, 0)),
    style=SdfLineStyle(color="#44ff44", thickness=0.15),
)
viz.add(Point(0, 0, 0), style=SdfPointStyle(color="#ff4444", size=0.15))
viz.add(
    Box(center=Point(2, 0, 0), size=(1.2, 1.2, 1.2)),
    style=SdfBoxStyle(color="#44aaff"),
)
viz.add(
    Ellipsoid(center=Point(4, 0, 0), radii=(1.0, 0.6, 0.8)),
    style=SdfEllipsoidStyle(color="#ff44ff"),
)

viz.display_snapshot()


## 3. `SdfObject` + Python CSG operators

`SdfObject` bundles a geometry entity, an optional `id`, and a per-entity style.
Combine objects with the Python CSG operators:

| Operator | Meaning |
|---|---|
| `+` / `|` | union |
| `-` | subtract |
| `&` | intersection |
| `^` | xor |
| `-a` | tag `a` with `SUBTRACT` polarity |
| `~a` | tag `a` with `INTERSECTION` polarity |

These are backed by the `ECompose`/`Combine` node model.


In [4]:
body = SdfObject(
    Sphere(Point(0, 0, 0), 1.2),
    id="body",
    style=SdfSphereStyle(color="#ffaa00"),
)
drill = SdfObject(
    Cylinder(
        origin=Point(0, -0.6, 0),
        axis=Direction(0, 1, 0),
        length=1.2,
        radius=0.35,
        align_center=0.5,
    ),
    id="drill",
    style=SdfCylinderStyle(color="#44ff44"),
)

viz = Visualizer(title="SDF — object model", add_default_axes=False, add_default_grid=False)
viz.add(body - drill, label="body − drill")
viz.display_snapshot()


## 4. `Composed` — one internally-CSG'd object

`Composed` bundles constituents (each with its own combine mode) into a **single**
object with one material. Constituents may be `SdfObject`s or SDF primitives
(`sphere`, `capped_cylinder`, …), and use the legacy `(obj, "subtract")` tuple
form for combine modes.


In [5]:
bead = Composed(
    sphere(0.7),
    (capped_cylinder(1.0, 0.45), "subtract"),
)

viz = Visualizer(title="SDF — Composed", add_default_axes=False, add_default_grid=False)
viz.add(bead, style=SdfStyle(color="#44ff44"), label="bead")
viz.display_snapshot()


## 5. `SdfGroup` — cross-object CSG + per-member transforms

`SdfGroup` bundles several members into **one** ray-marched solid, so
cross-object CSG (`union`/`intersection`/`subtract`), smooth shading, and
self-shadowing all work *across* members — while each member keeps an
independent runtime transform. Address members by `id` or 0-based index with
`set_member_transform(...)`. Up to 16 members are supported.


In [6]:
group = SdfGroup(
    sphere(1.0, position=(-1.0, 0.0, 0.0), id="left"),
    sphere(1.0, position=(1.0, 0.0, 0.0), id="orbit"),
    (capped_cylinder(1.5, 0.35), "subtract"),   # cut through both spheres
)

viz = Visualizer(title="SDF — group", add_default_axes=False, add_default_grid=False)
sdf_grp = viz.new(group, style=SdfStyle(color="#ffaa00"), label="SDF group")

# Move a member independently (by id or index); the proxy AABB resizes too.
sdf_grp.set_member_transform("orbit", position=(1.5, 0.4, 0.0))
viz.flush()
viz.display_snapshot()


## 6. Viz-only entities → SDF primitives

The visualization-only entities also map to SDF primitives, so every new solid
renders via `SdfObject(...)` too:

| Entity | SDF primitive | Slab style |
|---|---|---|
| `Disk` | `cappedCylinder` | `SdfDiskStyle(thickness=…)` |
| `PartialDisk` | `partial_disk` | `SdfPartialDiskStyle(thickness=…)` |
| `Box` | `box` | `SdfBoxStyle` |
| `Ellipsoid` / `Ellipse` | `ellipsoid` | `SdfEllipsoidStyle` / `SdfEllipseStyle` |
| `RegularPolygon` | `regular_polygon` | `SdfRegularPolygonStyle` |


In [7]:
viz = Visualizer(title="SDF — viz-only entities", add_default_axes=False, add_default_grid=False)

viz.add(SdfObject(Disk(center=Point(-3, 0, 0), radius=1.0), style=SdfDiskStyle(color="#ff8844", thickness=0.1)))
viz.add(SdfObject(PartialDisk(center=Point(-1, 0, 0), radius=1.0, angle=4.0), style=SdfPartialDiskStyle(color="#ffcc44", thickness=0.1)))
viz.add(SdfObject(Ellipse(center=Point(1, 0, 0), radius_u=1.0, radius_v=0.6), style=SdfEllipseStyle(color="#ff44ff", thickness=0.1)))
viz.add(SdfObject(regular_polygon(6, radius=1.0, center=Point(3, 0, 0)), style=SdfRegularPolygonStyle(color="#44ffaa", thickness=0.1)))

viz.display_snapshot()


## 7. Constraints

- **WebGL2 required** — SDF objects need GLSL3 + `gl_FragDepth`; on WebGL1 they
  are hidden and a single yellow warning banner is shown.
- **16-member cap** per `SdfGroup` (a compile-time uniform-array bound).
- **No cross-object shadows** — SDF objects only self-shadow within an
  object/group.

SDF objects otherwise support the same labels, interaction, and tweening as
meshes; frame-by-frame member animation uses the loop from
[Tutorial 13](../13_animation/).


## Visual Examples

A mixed scene — mesh sphere + SDF sphere + a `Composed` bead + an `SdfGroup` —
exported via `export_snapshot()`.


In [8]:
viz = Visualizer(title="SDF — mixed scene")

# Normal mesh objects
viz.add(Sphere(Point(-3.5, 0, 0), 1.0), color="#4477cc", label="mesh")
viz.add(Point(0, 0, 0), color="#ffffff", label="origin")

# SDF sphere
viz.add(Sphere(Point(-1.0, 0, 0), 1.1), style=SdfStyle(color="#ffaa00"), label="SDF sphere")

# Composed bead
bead = Composed(sphere(0.7), (capped_cylinder(1.0, 0.45), "subtract"))
viz.add(bead, style=SdfStyle(color="#44ff44"), label="bead")

# SdfGroup with a moved member
group = SdfGroup(
    sphere(1.0, position=(-1.0, 0, 0), id="left"),
    sphere(1.0, position=(1.0, 0, 0), id="orbit"),
    (capped_cylinder(1.5, 0.35), "subtract"),
)
sdf_grp = viz.new(group, style=SdfStyle(color="#ff88cc"), label="SDF group")
sdf_grp.set_member_transform("orbit", position=(1.5, 0.4, 0.0))
sdf_grp.translate(4, 0, 0)

viz.flush()
viz.display_snapshot()


## Summary

| Task | API |
|---|---|
| Opt one entity into SDF | `viz.add(entity, style=SdfStyle(color=...))` |
| Per-entity SDF style | `SdfSphereStyle(...)`, `SdfLineStyle(thickness=...)`, … |
| Compose + CSG | `SdfObject(...)`, then `+` / `-` / `&` / `^` |
| One internally-CSG'd object | `Composed(part, (part, "subtract"), …)` |
| Cross-object CSG + members | `SdfGroup(member, …)` + `set_member_transform(...)` |

**Next:** [06 — Multi-Scene](../06_multi_scene/).
